# 4 모델 구조 비교 (Ensemble vs Cascade vs Multi-task vs Multi-class)

**목적**: 같은 데이터로 4가지 학습 → 21 케이스 + test set 정확도 비교 → best 선정

**데이터**: dataset_v3_train/test.csv (29K, 4 클래스, manipulation 6% 불균형)

**비교 대상**:
1. **Ensemble**: 4 binary 분류기 (각 클래스 vs 나머지)
2. **Cascade**: manipulation binary 1차 → 3-class 2차
3. **Multi-task**: shared encoder + 4 binary heads
4. **Multi-class**: 1 모델, 4 클래스

**사용법**:
1. Colab → 파일 → 노트북 업로드
2. 런타임 → T4 GPU
3. 좌측 패널에 업로드:
   - `dataset_v3_train.csv`
   - `dataset_v3_test.csv`
   - `sequences.jsonl`
4. 모두 실행 (~1~2시간 예상)
5. 마지막에 비교 표 + 모델들 zip 다운로드

## 1. 환경 + 의존성

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('!!! 런타임 → T4 GPU로 변경하세요')

In [ ]:
!pip install -q transformers datasets scikit-learn pandas

## 2. 데이터 + 토크나이저 (공통)

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
import torch.nn as nn
from torch.optim import AdamW
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

for f in ['dataset_v3_train.csv', 'dataset_v3_test.csv', 'sequences.jsonl']:
    assert os.path.exists(f), f'{f} 누락'
    print(f'OK: {f} ({os.path.getsize(f)/1024:.1f} KB)')

train_df = pd.read_csv('dataset_v3_train.csv')
test_df = pd.read_csv('dataset_v3_test.csv')

LABEL_NAMES = {0: 'normal', 1: 'positive', 2: 'vulnerable', 3: 'manipulation'}
NUM_CLASSES = 4

print(f'\ntrain: {len(train_df)}')
for lbl, cnt in sorted(train_df.label.value_counts().items()):
    print(f'  {lbl} {LABEL_NAMES[lbl]}: {cnt}')
print(f'\ntest: {len(test_df)}')
for lbl, cnt in sorted(test_df.label.value_counts().items()):
    print(f'  {lbl} {LABEL_NAMES[lbl]}: {cnt}')

# 21 케이스 (OOD)
with open('sequences.jsonl', 'r', encoding='utf-8') as f:
    cases = [json.loads(line) for line in f if line.strip()]
eval_turns = []
for case in cases:
    is_manip = not case['case_id'].startswith('normal_')
    for turn in case['turns']:
        if turn['role'] == 'user':
            eval_turns.append({
                'case_id': case['case_id'],
                'category': case['category'],
                'text': turn['content'],
                'expected_binary': 1 if is_manip else 0,  # manipulation=1
            })
print(f'\n21 케이스 user turns: {len(eval_turns)}')

In [ ]:
MODEL_NAME = 'klue/roberta-base'
MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 3
LR = 2e-5
device = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# class weights (inverse frequency)
counts = np.array([train_df[train_df.label == i].shape[0] for i in range(NUM_CLASSES)])
class_weights = torch.tensor(
    len(train_df) / (NUM_CLASSES * counts), dtype=torch.float
).to(device)
print(f'class weights (inverse freq): {class_weights.tolist()}')

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=MAX_LEN, padding='max_length', truncation=True, return_tensors='pt')
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long),
        }

def make_loader(df, label_col='label', shuffle=True):
    ds = TextDataset(df.text.tolist(), df[label_col].tolist(), tokenizer)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

test_loader_full = make_loader(test_df, shuffle=False)
print(f'test batches: {len(test_loader_full)}')

## 3. 공통 학습/평가 함수

In [ ]:
def train_classifier(model, train_loader, num_labels, epochs=EPOCHS, class_weight=None):
    model.to(device)
    optimizer = AdamW(model.parameters(), lr=LR)
    if class_weight is not None:
        criterion = nn.CrossEntropyLoss(weight=class_weight)
    else:
        criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        pbar = tqdm(train_loader, desc=f'epoch {epoch+1}/{epochs}')
        for batch in pbar:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            out = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
            loss = criterion(out.logits, batch['labels'])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix(loss=f'{loss.item():.3f}')
        print(f'  epoch {epoch+1} avg loss: {total_loss/len(train_loader):.4f}')
    return model

def predict_probs(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
            probs = torch.softmax(out.logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
            all_labels.extend(batch['labels'].cpu().numpy().tolist())
    return np.concatenate(all_probs), np.array(all_labels)

def predict_text_probs(model, texts, num_outputs):
    """단발 텍스트 리스트 → prob (num_outputs)"""
    model.eval()
    probs_list = []
    with torch.no_grad():
        for text in texts:
            enc = tokenizer(text, max_length=MAX_LEN, padding='max_length', truncation=True, return_tensors='pt').to(device)
            out = model(input_ids=enc['input_ids'], attention_mask=enc['attention_mask'])
            probs = torch.softmax(out.logits, dim=-1).cpu().numpy()[0]
            probs_list.append(probs)
    return np.array(probs_list)

## 4. 모델 A: Multi-class (1 모델, 4 클래스)

In [ ]:
print('=' * 60)
print('A. Multi-class (4 classes)')
print('=' * 60)
model_mc = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_CLASSES)
train_loader = make_loader(train_df)
model_mc = train_classifier(model_mc, train_loader, NUM_CLASSES, class_weight=class_weights)

In [ ]:
# Test 평가
probs_mc_test, labels_mc_test = predict_probs(model_mc, test_loader_full)
preds_mc_test = probs_mc_test.argmax(axis=1)
test_acc_mc = accuracy_score(labels_mc_test, preds_mc_test)
test_f1_mc = f1_score(labels_mc_test, preds_mc_test, average='macro')
print(f'Multi-class TEST: acc={test_acc_mc:.4f}, macro F1={test_f1_mc:.4f}')
print(classification_report(labels_mc_test, preds_mc_test, target_names=[LABEL_NAMES[i] for i in range(NUM_CLASSES)], digits=4))

## 5. 모델 B: Cascade (manipulation 1차 → 3-class 2차)

In [ ]:
print('=' * 60)
print('B. Cascade (manipulation binary → 3-class)')
print('=' * 60)

# Stage 1: manipulation binary
train_df_stage1 = train_df.copy()
train_df_stage1['stage1_label'] = (train_df_stage1['label'] == 3).astype(int)  # 1=manip, 0=else
loader_stage1 = make_loader(train_df_stage1, label_col='stage1_label')

n1 = (train_df_stage1.stage1_label == 1).sum()
n0 = (train_df_stage1.stage1_label == 0).sum()
weights_stage1 = torch.tensor([len(train_df_stage1)/(2*n0), len(train_df_stage1)/(2*n1)], dtype=torch.float).to(device)

model_cascade_s1 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
print('Stage 1 (manipulation binary) 학습...')
model_cascade_s1 = train_classifier(model_cascade_s1, loader_stage1, 2, class_weight=weights_stage1)

In [ ]:
# Stage 2: 3-class (normal/positive/vulnerable, manipulation 제외)
train_df_stage2 = train_df[train_df['label'] != 3].copy()
# 라벨 remap: 0→0, 1→1, 2→2
loader_stage2 = make_loader(train_df_stage2)

counts2 = np.array([train_df_stage2[train_df_stage2.label == i].shape[0] for i in range(3)])
weights_stage2 = torch.tensor(len(train_df_stage2)/(3*counts2), dtype=torch.float).to(device)

model_cascade_s2 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
print('Stage 2 (3-class) 학습...')
model_cascade_s2 = train_classifier(model_cascade_s2, loader_stage2, 3, class_weight=weights_stage2)

In [ ]:
# Cascade inference
def predict_cascade(model_s1, model_s2, loader):
    """Stage 1 → Stage 2 라우팅, 최종 라벨 0~3."""
    model_s1.eval(); model_s2.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            out1 = model_s1(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
            probs_s1 = torch.softmax(out1.logits, dim=-1)
            is_manip = probs_s1[:, 1] > 0.5
            
            out2 = model_s2(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
            preds_s2 = out2.logits.argmax(dim=-1)  # 0, 1, 2
            
            final = torch.where(is_manip, torch.tensor(3, device=device), preds_s2)
            all_preds.extend(final.cpu().numpy().tolist())
            all_labels.extend(batch['labels'].cpu().numpy().tolist())
    return np.array(all_preds), np.array(all_labels)

preds_cascade, labels_cascade = predict_cascade(model_cascade_s1, model_cascade_s2, test_loader_full)
test_acc_cascade = accuracy_score(labels_cascade, preds_cascade)
test_f1_cascade = f1_score(labels_cascade, preds_cascade, average='macro')
print(f'Cascade TEST: acc={test_acc_cascade:.4f}, macro F1={test_f1_cascade:.4f}')
print(classification_report(labels_cascade, preds_cascade, target_names=[LABEL_NAMES[i] for i in range(NUM_CLASSES)], digits=4))

## 6. 모델 C: Ensemble (4 binary 분류기)

In [ ]:
print('=' * 60)
print('C. Ensemble (4 binary, each: class vs rest)')
print('=' * 60)

ensemble_models = {}
for class_idx in range(NUM_CLASSES):
    name = LABEL_NAMES[class_idx]
    print(f'\n--- {class_idx} {name} binary 학습 ---')
    df_b = train_df.copy()
    df_b['binary_label'] = (df_b['label'] == class_idx).astype(int)
    loader_b = make_loader(df_b, label_col='binary_label')
    
    n1 = (df_b.binary_label == 1).sum()
    n0 = (df_b.binary_label == 0).sum()
    w = torch.tensor([len(df_b)/(2*n0), len(df_b)/(2*n1)], dtype=torch.float).to(device)
    
    m = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    m = train_classifier(m, loader_b, 2, class_weight=w)
    ensemble_models[class_idx] = m

In [ ]:
# Ensemble inference: 각 모델의 prob_class1 → argmax
def predict_ensemble(models, loader):
    for m in models.values():
        m.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            probs = torch.zeros(batch['input_ids'].size(0), NUM_CLASSES, device=device)
            for class_idx, m in models.items():
                out = m(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
                p = torch.softmax(out.logits, dim=-1)[:, 1]  # prob of class=1
                probs[:, class_idx] = p
            preds = probs.argmax(dim=-1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(batch['labels'].cpu().numpy().tolist())
    return np.array(all_preds), np.array(all_labels)

preds_ens, labels_ens = predict_ensemble(ensemble_models, test_loader_full)
test_acc_ens = accuracy_score(labels_ens, preds_ens)
test_f1_ens = f1_score(labels_ens, preds_ens, average='macro')
print(f'Ensemble TEST: acc={test_acc_ens:.4f}, macro F1={test_f1_ens:.4f}')
print(classification_report(labels_ens, preds_ens, target_names=[LABEL_NAMES[i] for i in range(NUM_CLASSES)], digits=4))

## 7. 모델 D: Multi-task (shared encoder + 4 heads)

In [ ]:
print('=' * 60)
print('D. Multi-task (shared encoder + 4 binary heads)')
print('=' * 60)

class MultiTaskModel(nn.Module):
    def __init__(self, base_name, num_tasks=NUM_CLASSES):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_name)
        h = self.encoder.config.hidden_size
        self.heads = nn.ModuleList([nn.Linear(h, 1) for _ in range(num_tasks)])
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]  # CLS
        logits = torch.cat([h(cls) for h in self.heads], dim=-1)  # (B, num_tasks)
        return logits

mt_model = MultiTaskModel(MODEL_NAME).to(device)

# 학습용 멀티 라벨 (one-hot)
class MTDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=MAX_LEN, padding='max_length', truncation=True, return_tensors='pt')
        # one-hot multi-label (단일 클래스만 1)
        one_hot = torch.zeros(NUM_CLASSES, dtype=torch.float)
        one_hot[self.labels[idx]] = 1.0
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels': one_hot,
        }

mt_train_loader = DataLoader(
    MTDataset(train_df.text.tolist(), train_df.label.tolist(), tokenizer),
    batch_size=BATCH_SIZE, shuffle=True
)
mt_test_loader = DataLoader(
    MTDataset(test_df.text.tolist(), test_df.label.tolist(), tokenizer),
    batch_size=BATCH_SIZE, shuffle=False
)

# class-wise pos_weight (불균형 처리)
pos_w = torch.tensor([(counts.sum()-c)/c for c in counts], dtype=torch.float).to(device)
mt_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
mt_optimizer = AdamW(mt_model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    mt_model.train()
    total_loss = 0
    pbar = tqdm(mt_train_loader, desc=f'MT epoch {epoch+1}/{EPOCHS}')
    for batch in pbar:
        batch = {k: v.to(device) for k, v in batch.items()}
        mt_optimizer.zero_grad()
        logits = mt_model(batch['input_ids'], batch['attention_mask'])
        loss = mt_criterion(logits, batch['labels'])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(mt_model.parameters(), 1.0)
        mt_optimizer.step()
        total_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.3f}')
    print(f'  MT epoch {epoch+1} avg loss: {total_loss/len(mt_train_loader):.4f}')

In [ ]:
# Multi-task eval
mt_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in mt_test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = mt_model(batch['input_ids'], batch['attention_mask'])
        probs = torch.sigmoid(logits)
        preds = probs.argmax(dim=-1)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(batch['labels'].argmax(dim=-1).cpu().numpy().tolist())

test_acc_mt = accuracy_score(all_labels, all_preds)
test_f1_mt = f1_score(all_labels, all_preds, average='macro')
print(f'Multi-task TEST: acc={test_acc_mt:.4f}, macro F1={test_f1_mt:.4f}')
print(classification_report(all_labels, all_preds, target_names=[LABEL_NAMES[i] for i in range(NUM_CLASSES)], digits=4))

## 8. 21 케이스 OOD 평가 (binary: manipulation vs rest)

In [ ]:
case_texts = [t['text'] for t in eval_turns]
case_expected = np.array([t['expected_binary'] for t in eval_turns])  # 1=manipulation

def evaluate_on_cases(name, predict_fn):
    """predict_fn(texts) → preds (0~3). manipulation=3 → binary 1"""
    preds = predict_fn(case_texts)
    pred_binary = (preds == 3).astype(int)
    acc = (pred_binary == case_expected).mean()
    print(f'{name} 21 케이스 binary acc: {acc:.3f} ({(pred_binary == case_expected).sum()}/{len(case_expected)})')
    # 미분류 case 보여주기
    for i, (t, exp, pr_b, pr_full) in enumerate(zip(eval_turns, case_expected, pred_binary, preds)):
        if pr_b != exp:
            print(f'  ❌ {t["case_id"]:38s} | pred={LABEL_NAMES[pr_full]}, expected_binary={exp}')
            print(f'      user: {t["text"][:80]}')
    return acc

# Multi-class
def predict_mc(texts):
    probs = predict_text_probs(model_mc, texts, NUM_CLASSES)
    return probs.argmax(axis=1)

# Cascade
def predict_cas(texts):
    probs_s1 = predict_text_probs(model_cascade_s1, texts, 2)
    is_manip = probs_s1[:, 1] > 0.5
    probs_s2 = predict_text_probs(model_cascade_s2, texts, 3)
    preds_s2 = probs_s2.argmax(axis=1)  # 0/1/2
    final = np.where(is_manip, 3, preds_s2)
    return final

# Ensemble
def predict_ens(texts):
    all_probs = np.zeros((len(texts), NUM_CLASSES))
    for class_idx, m in ensemble_models.items():
        probs = predict_text_probs(m, texts, 2)
        all_probs[:, class_idx] = probs[:, 1]
    return all_probs.argmax(axis=1)

# Multi-task
def predict_mt(texts):
    mt_model.eval()
    preds = []
    with torch.no_grad():
        for text in texts:
            enc = tokenizer(text, max_length=MAX_LEN, padding='max_length', truncation=True, return_tensors='pt').to(device)
            logits = mt_model(enc['input_ids'], enc['attention_mask'])
            probs = torch.sigmoid(logits).cpu().numpy()[0]
            preds.append(probs.argmax())
    return np.array(preds)

print('=' * 60)
ood_mc = evaluate_on_cases('Multi-class', predict_mc); print()
ood_cas = evaluate_on_cases('Cascade', predict_cas); print()
ood_ens = evaluate_on_cases('Ensemble', predict_ens); print()
ood_mt = evaluate_on_cases('Multi-task', predict_mt); print()

## 9. 최종 비교 표

In [ ]:
results = pd.DataFrame({
    '구조': ['Multi-class', 'Cascade', 'Ensemble', 'Multi-task'],
    '모델 수': [1, 2, 4, '1+heads'],
    'Test acc': [test_acc_mc, test_acc_cascade, test_acc_ens, test_acc_mt],
    'Test macro F1': [test_f1_mc, test_f1_cascade, test_f1_ens, test_f1_mt],
    '21케이스 binary acc': [ood_mc, ood_cas, ood_ens, ood_mt],
})
print('\n비교 표:')
print(results.to_string(index=False))

# best 추천
best_idx = results['21케이스 binary acc'].idxmax()
print(f'\nBest by 21케이스: {results.iloc[best_idx]["구조"]} ({results.iloc[best_idx]["21케이스 binary acc"]:.3f})')
best_idx_f1 = results['Test macro F1'].idxmax()
print(f'Best by Test macro F1: {results.iloc[best_idx_f1]["구조"]} ({results.iloc[best_idx_f1]["Test macro F1"]:.4f})')

## 10. 저장 + 다운로드

In [ ]:
import shutil

# 결과 CSV
results.to_csv('compare_v3_results.csv', index=False, encoding='utf-8-sig')

# 모델별 저장
model_mc.save_pretrained('./mc_v3'); tokenizer.save_pretrained('./mc_v3')
model_cascade_s1.save_pretrained('./cascade_s1'); tokenizer.save_pretrained('./cascade_s1')
model_cascade_s2.save_pretrained('./cascade_s2'); tokenizer.save_pretrained('./cascade_s2')
for ci, m in ensemble_models.items():
    m.save_pretrained(f'./ensemble_{LABEL_NAMES[ci]}')
    tokenizer.save_pretrained(f'./ensemble_{LABEL_NAMES[ci]}')
# Multi-task — 인코더 + heads state
os.makedirs('./multitask', exist_ok=True)
torch.save(mt_model.state_dict(), './multitask/model.pt')
tokenizer.save_pretrained('./multitask')

# 각자 zip
for d in ['mc_v3', 'cascade_s1', 'cascade_s2', 'multitask'] + [f'ensemble_{LABEL_NAMES[i]}' for i in range(NUM_CLASSES)]:
    shutil.make_archive(d, 'zip', d)
    print(f'zip: {d}.zip')

In [ ]:
from google.colab import files
files.download('compare_v3_results.csv')
# best 모델만 다운로드 (메모리 절약)
best_name = results.iloc[best_idx]['구조']
if best_name == 'Multi-class':
    files.download('mc_v3.zip')
elif best_name == 'Cascade':
    files.download('cascade_s1.zip')
    files.download('cascade_s2.zip')
elif best_name == 'Ensemble':
    for i in range(NUM_CLASSES):
        files.download(f'ensemble_{LABEL_NAMES[i]}.zip')
elif best_name == 'Multi-task':
    files.download('multitask.zip')

print(f'\n전체 비교 결과: compare_v3_results.csv')
print(f'Best 구조: {best_name}')